# eur_r2 — covariate PCs

The 20 PCs carried as covariates into residualization, following Kemper et al.:
HapMap3 common variants, LD-pruned, PCA fit on the analysis sample.

Two decisions, settled by the variant-set exploration (now in git history,
`analyses/eur_pipeline/01b_variant_set_exploration_cells.md` at `2d5c9c0`):

- **HM3 common only.** Non-HM3 arms are not projectable against 1000G and added
  little.
- **r²=0.1, not 0.05.** At 0.05 the HM3 common set loses 77% of its variants;
  0.1 yields ~35–40K and still removes most LD. The gate prunes harder because
  it needs clean axes, not coverage; the covariates want the opposite.

QC is re-run here from `r1_qc` rather than reusing the gate's pruned panel — same
filters, but the covariate PCA needs the unpruned variant pool.

**Runs:** entirely on this VM. `--pca approx` handles ~223K samples.
**Reads:** `r1_qc` and the keep list from `gate.md`.
**Writes:** `R/01_ancestry/covariate_pca/`.

## setup

In [ ]:
import os, sys
sys.path.insert(0, os.path.expanduser("~/aou-covariance/runs/_lib"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from aoucov import Run
from aoucov import checks, env, plink, plots, provenance, refs

run = Run("eur_r2")
GATE_OUT = run.out("01_ancestry", "round2")
OUT = run.out("01_ancestry", "covariate_pca")
LOCAL = run.scratch("covpca")

R1_QC = f"{LOCAL}/r1_qc"
KEEP = f"{GATE_OUT}/eur_r2_keep_ids.txt"

PRUNE = "1000kb 1 0.1"
N_PCS_FIT = 20
N_PCS_COVARIATE = 20
META = None    # TSV with person_id + data_partner_id etc., when one exists

refs.assert_inputs()
assert os.path.isfile(KEEP), f"{KEEP} — run gate.md first"
print(f"keep:   {sum(1 for _ in open(KEEP)):,} participants")
print(f"output: {OUT}")

env.ensure_plink2()
env.sh(f'gsutil -m cp "{run.out_gs("01_ancestry", "round2")}/panels/r1_qc."* "{LOCAL}/"')

## QC in the round-2 cohort, and the HM3 list

In [ ]:
r2_qc = plink.qc(R1_QC, f"{LOCAL}/r2_qc", keep=KEEP)
hm3 = refs.shared_variants(f"{r2_qc}.pvar", f"{LOCAL}/hm3.ids")

## prune

In [ ]:
ld = refs.write_ld_regions(f"{LOCAL}/high_ld_regions.txt", refs.HIGH_LD_CANONICAL)
pruned = plink.prune(r2_qc, f"{LOCAL}/hm3_common", extract=hm3,
                     ld_regions=ld, params=PRUNE, maf=0.01)

## fit and score

Everyone is scored through the fit's own loadings, with the fit's `.acount` as
the frequency reference, so the 1000G projection below lands on the same axes.

In [ ]:
pca = plink.pca(r2_qc, f"{LOCAL}/covpca", extract=f"{pruned}.prune.in",
                n_pcs=N_PCS_FIT)
W, FREQ = f"{pca}.eigenvec.allele", f"{pca}.acount"

plink.score(W, f"{LOCAL}/part_covpca", pfile=r2_qc, freq=FREQ,
            extract=f"{pruned}.prune.in", n_pcs=N_PCS_FIT)
part = plink.read_scores(f"{LOCAL}/part_covpca.sscore", "person_id", N_PCS_FIT)

kg_ids = refs.shared_variants(W, f"{LOCAL}/covpca_kg.ids", cols="2,4,5")
plink.score(W, f"{LOCAL}/kg_covpca", bfile=refs.KG_BFILE, freq=FREQ,
            extract=kg_ids, n_pcs=N_PCS_FIT)
kg = plink.read_scores(f"{LOCAL}/kg_covpca.sscore", "sample", N_PCS_FIT).merge(
    refs.panel(), on="sample", how="left")

_, pct = plink.read_eigenval(pca)
print(f"{len(part):,} participants; PC1 {pct[0]:.2f}%  PC2 {pct[1]:.2f}%  "
      f"PC{N_PCS_FIT} {pct[N_PCS_FIT-1]:.2f}%")

## structure the PCs capture

In [ ]:
plots.scree(pca, f"{OUT}/covpca_scree.png", title="eur_r2 covariate PCA")
plots.subpop_panel(part, kg, f"{OUT}/covpca_pc_pairs.png", n_pairs=4,
                   n_pcs=N_PCS_FIT,
                   title=f"eur_r2 covariate PCA — HM3 common, r²=0.1, n={len(part):,}")

sep = plots.anchor_separation(kg, part, N_PCS_FIT)
print(pd.DataFrame({"PC": plink.pc_names(N_PCS_FIT), "pct_var": pct.round(2),
                    "CEU+GBR vs rest of EUR": sep.round(2)}).to_string(index=False))

## are these PCs real?

An LD peak means a PC is tracking one locus; a high batch R² means it is tracking
the assay. Either is still usable as a covariate, but not as ancestry.

In [ ]:
L = plots.loadings_by_position(pca, f"{OUT}/covpca_loadings.png", n_show=4,
                               title="eur_r2 covariate PCA — loadings")
peaks = checks.ld_peaks(L, n_pcs=10)
checks.batch_effect(part, META)

If `peaks` is non-empty: add them to the exclusion list and rerun from the prune
step.

## write

In [ ]:
cov = part[["person_id"] + plink.pc_names(N_PCS_COVARIATE)].rename(
    columns={"person_id": "IID"})
COV_PATH = f"{OUT}/covariate_pcs_eur_r2.txt"
cov.to_csv(COV_PATH, sep="\t", index=False)

n_variants = sum(1 for _ in open(f"{pruned}.prune.in"))
provenance.write(OUT, "covariate_pca", {
    "input": "r1_qc, re-QCd in the round-2 cohort",
    "variants": "HM3 common (maf>=0.01, ID+REF+ALT match against 1000G)",
    "prune": f"--indep-pairwise {PRUNE}, long-range LD regions excluded",
    "n_variants_pruned": n_variants,
    "pca": f"--pca approx {N_PCS_FIT} allele-wts, fit on the round-2 participants",
    "scores": "--score through the fit's loadings, --read-freq its .acount",
    "covariates": f"PC1-{N_PCS_COVARIATE}",
    "n_participants": len(cov),
})
provenance.result_line("covariate_pcs", n=len(cov), n_variants=n_variants,
                       pc1_pct=round(pct[0], 2))
print(f"{len(cov):,} participants, PC1-{N_PCS_COVARIATE} -> {COV_PATH}")

Next: residualization reads `covariate_pcs_eur_r2.txt`.